# STEP 3. Label creation + sample filter + log transform

## Label definition

```
net_change(t) = opening_count(t) − closure_count(t)
y(t)          = 1 if net_change(t+1) < 0 else 0     ← switch to net decline next quarter
```

**Two cautions**

- `shift(-1)` must be done within each (commercial area, service category) group **after sorting by quarter ascending**. Otherwise the last-quarter label of commercial area A gets the first-quarter value of commercial area B
- If the observed quarters are not consecutive, no label is created. `shift(-1)` only fetches the "next in row order", not the "next in quarter terms"

## Role of 20261

13 quarters were loaded, but **20261 is there to supply labels**. Building `y` for 20254 requires the openings and closures of 20261. 20261 itself has no next quarter, so it cannot get a label and drops out here.

→ Training window **20231 ~ 20254, 12 quarters**

## Filter rationale

| Filter | Reason |
|---|---|
| Label exists | Cannot train without y |
| `store_count` ≥ 5 | In a cell with 3-4 stores, a single opening and a single closure flip the label. More noise than signal |
| Foot traffic exists | H2 matching covariate ②; matching is impossible without it |
| Area > 0 | Outlier for which density cannot be computed |

> ⚠️ **Key change in this revision** — the 5-or-more filter is now based on `store_count` (total count), not `nonfranchise_store_count` (non-franchise). More cells pass, and **franchise-dense service categories come in especially often.** Section 4-2 checks that change with numbers.

In [1]:
import sys, os
from pathlib import Path

# Find the project root wherever the notebook is opened and add it to sys.path
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
print("Project root:", ROOT.name)

Project root: seoul-closure-risk


In [2]:
from config import (PROC, MIN_STORES, QUARTERS, COVARIATES,
                    TRAIN_QUARTERS, VALID_Q, SCORE_Q)

df = pd.read_pickle(PROC / "02_merged.pkl")
assert "store_count" in df.columns, "Re-run STEP 2 with the new version"

def quarter_index(code):
    """20241 -> consecutive integer. For judging quarter continuity"""
    return (code // 10) * 4 + (code % 10 - 1)

df["quarter_idx"]    = quarter_index(df["quarter_code"])
df["net_change"]      = df["opening_count"] - df["closure_count"]
df["net_decline_now"] = (df["net_change"] < 0).astype(int)

print(f"Input {df.shape}")
print(f"{len(QUARTERS)} quarters (the last, {QUARTERS[-1]}, is for supplying labels)")

Input (160352, 26)
13 quarters (the last, 20261, is for supplying labels)


## 3-1. Label creation

In [3]:
assert not df.duplicated(["area_code", "service_code", "quarter_idx"]).any(), "Sort key is not unique"
df = df.sort_values(["area_code", "service_code", "quarter_idx"], kind="mergesort").reset_index(drop=True)
g = df.groupby(["area_code", "service_code"], sort=False)

df["y"]          = g["net_decline_now"].shift(-1)
df["next_quarter_idx"] = g["quarter_idx"].shift(-1)

streak = (df["next_quarter_idx"] - df["quarter_idx"]) == 1
gap_found = df["y"].notna() & ~streak
df.loc[~streak, "y"] = np.nan

print(f"Labels created {df['y'].notna().sum():,} / total {len(df):,}")
print(f"  └ Labels invalidated by quarter gaps: {gap_found.sum():,}\n")
display(df.groupby("quarter_code")["y"].agg(n_rows="size", has_label="count")
        .assign(label_rate=lambda t: (t["has_label"]/t["n_rows"]).round(3)))
print(f"※ A label rate of 0 for {QUARTERS[-1]} is normal (no next quarter)")

Labels created 147,138 / total 160,352
  └ Labels invalidated by quarter gaps: 140



,n_rows,has_label,label_rate
quarter_code,,,
20231,12397,12315,0.993
20232,12409,12310,0.992
20233,12393,12312,0.993
20234,12405,12334,0.994
20241,12418,12325,0.993
20242,12388,12296,0.993
20243,12348,12263,0.993
20244,12320,12245,0.994
20251,12304,12236,0.994


※ A label rate of 0 for 20261 is normal (no next quarter)


## 3-2. Sample filter — tracking losses step by step

In [4]:
steps = [("Original", df)]
d = df[df["y"].notna()].copy();               steps.append(("Label exists", d))
d = d[d["store_count"] >= MIN_STORES];        steps.append((f"{MIN_STORES}+ stores", d))
d = d[d["foot_traffic_total"].notna()];             steps.append(("Foot traffic exists", d))
d = d[d["area_size"] > 0];                     steps.append(("Area valid", d))

prev = None
for nm, s in steps:
    drop = "" if prev is None else f"  (-{prev-len(s):,})"
    print(f"  {nm:14} {len(s):>8,}{drop}")
    prev = len(s)

d["y"] = d["y"].astype(int)
print(f"\nFinal sample {len(d):,} rows | positive rate (net decline) {d['y'].mean():.1%}")
print(f"Training window {d['quarter_code'].min()} ~ {d['quarter_code'].max()} "
      f"({d['quarter_code'].nunique()} quarters)")
print(f"Share of no-event cells (opening=0 & closure=0): "
      f"{((d['opening_count']==0)&(d['closure_count']==0)).mean():.1%}")

  Original        160,352
  Label exists    147,138  (-13,214)
  5+ stores        67,495  (-79,643)
  Foot traffic exists   67,475  (-20)
  Area valid       67,475  (-0)

Final sample 67,475 rows | positive rate (net decline) 25.5%
Training window 20231 ~ 20254 (12 quarters)
Share of no-event cells (opening=0 & closure=0): 43.7%


## 3-3. Impact of changing the store-count criterion ★

The previous pipeline filtered on 5 or more using `nonfranchise_store_count` (non-franchise). We check which cells newly enter when the criterion switches to the total count.

**If the incoming cells are concentrated in particular service categories**, the sample composition of the earlier analysis was itself asymmetric across service categories.

In [5]:
base = df[df["y"].notna() & df["foot_traffic_total"].notna() & (df["area_size"] > 0)]
old = base["nonfranchise_store_count"] >= MIN_STORES     # previous criterion
new = base["store_count"] >= MIN_STORES     # current criterion

print(f"Passing previous criterion (nonfranchise_store_count): {old.sum():,}")
print(f"Passing current criterion (store_count)             : {new.sum():,}   (+{new.sum()-old.sum():,}, "
      f"{(new.sum()-old.sum())/old.sum():+.1%})\n")

added = base[new & ~old]
t = (added.groupby("service_name")
     .agg(new_entry=("y", "size"), net_decline_rate=("y", "mean"))
     .join(base[old].groupby("service_name").size().rename("prior_sample"))
     .assign(growth_rate=lambda x: (x["new_entry"]/x["prior_sample"]))
     .sort_values("growth_rate", ascending=False))
display(t.round(3))
print("※ Service categories with a high growth rate = those with a high franchise share that were under-counted before")
print(f"※ Net decline rate of newly entering cells {added['y'].mean():.1%} vs existing {base[old]['y'].mean():.1%}")

Passing previous criterion (nonfranchise_store_count): 55,795
Passing current criterion (store_count)             : 67,475   (+11,680, +20.9%)



,new_entry,net_decline_rate,prior_sample,growth_rate
service_name,,,,
Fried-chicken shops,3332,0.189,854,3.902
Fast-food restaurants,2214,0.189,1687,1.312
Bakeries,1145,0.151,2988,0.383
Korean snack-food restaurants,1344,0.167,6226,0.216
Coffee and beverages,1563,0.143,10483,0.149
Chinese restaurants,420,0.152,3226,0.130
Japanese restaurants,403,0.154,3293,0.122
Western restaurants,345,0.180,3827,0.090
Beer and casual pubs,572,0.119,7876,0.073


※ Service categories with a high growth rate = those with a high franchise share that were under-counted before
※ Net decline rate of newly entering cells 16.9% vs existing 27.3%


## 3-4. Sales availability — analysis sample basis

In the full panel 45% of sales are missing, but most of that is non-disclosure for cells with 2 or fewer stores. **In the analysis sample that passed the 5-or-more filter, the situation is different.**

In [6]:
d["sales_observed"] = d["monthly_sales"].notna()   # also created in add_features, but used here first
print(f"Sales observed rate in analysis sample {d['sales_observed'].mean():.1%}")
print(f"  Neighborhood commercial areas only {d[d['area_type']=='neighborhood commercial area']['sales_observed'].mean():.1%}\n")

display(d.groupby("service_name")
        .agg(observed_rate=("sales_observed", "mean"), n_cells=("sales_observed", "size"))
        .sort_values("observed_rate", ascending=False).round(3))
print("※ The observed rate differs by service category → putting sales into the matching covariates would selectively drop only control cells.")
print("   It is used in secondary question 2 (STEP 7), but excluded from the main PSM analysis and handled as a robustness analysis.")

Sales observed rate in analysis sample 92.9%
  Neighborhood commercial areas only 90.9%



,observed_rate,n_cells
service_name,,
Korean restaurants,0.988,15677
Beer and casual pubs,0.986,8448
Korean snack-food restaurants,0.946,7570
Chinese restaurants,0.938,3646
Coffee and beverages,0.927,12046
Fried-chicken shops,0.913,4186
Japanese restaurants,0.873,3696
Fast-food restaurants,0.842,3901
Bakeries,0.839,4133


※ The observed rate differs by service category → putting sales into the matching covariates would selectively drop only control cells.
   It is used in secondary question 2 (STEP 7), but excluded from the main PSM analysis and handled as a robustness analysis.


## 3-5. Positive rate by quarter

If only a particular quarter stands out, it is seasonality or a problem in the source data.

In [7]:
display(d.groupby("quarter_code")
        .agg(n_cells=("y", "size"), net_decline_rate=("y", "mean"),
             mean_stores=("store_count", "mean")).round(3))

,n_cells,net_decline_rate,mean_stores
quarter_code,,,
20231,5706,0.229,21.067
20232,5731,0.261,21.101
20233,5701,0.223,21.168
20234,5730,0.236,21.198
20241,5706,0.254,21.398
20242,5678,0.276,21.488
20243,5622,0.262,21.547
20244,5586,0.284,21.612
20251,5534,0.261,21.558


## 3-6. Log transform

`area_size` ranges from a minimum of 1,854㎡ to a maximum of 2,462,734㎡, a **1,300x** difference. The store count also has a long one-sided tail. Left on the original scale, a few extreme values would dominate the mean and the matching-distance computation.

> ⚠️ **The definition of `area_size` is missing from the source documentation.**
> Given that the median area per store is in the 400㎡ range and the largest commercial area is 2.46km², it looks like **the total area of the commercial-area polygon, not the commercial-use floor area**. However, this is an inference from the numbers, not a confirmed fact.
> Since this variable is **one of the 4 PSM covariates**, it must be stated as a limitation. Fortunately, in the STEP 9 counterfactual, fixing only the area barely moved the gap (+3.20 percentage points (pp) vs. the original +3.07pp), so the conclusion does not depend on this variable.

### On store density

We create `store_density = store_count / area_size`, but **do not add it as a matching covariate.**

```
log(density) = log(store count) − log(area)
```

It is a linear combination of two variables that are already being matched, so if their means are balanced, the mean of log density is automatically balanced too. Adding it as a separate variable would create perfect collinearity. It is kept for interpretation and descriptive statistics only.

In [8]:
def add_features(f):
    """Log transforms and derived variables. Applied identically to the training panel and the prediction frame."""
    f = f.copy()
    f["log_store_count"]   = np.log(f["store_count"])
    f["log_foot_traffic"] = np.log1p(f["foot_traffic_total"])
    f["log_area_size"]     = np.log(f["area_size"])
    f["log_attracting_facilities"] = np.log1p(f["attracting_facility_count"].fillna(0))
    # derived (not a matching covariate)
    f["store_density"]   = f["store_count"] / f["area_size"] * 1000   # stores per 1,000㎡
    f["sales_per_store"] = f["monthly_sales"] / f["store_count"]
    f["sales_observed"]  = f["monthly_sales"].notna()
    return f

d = add_features(d)
display(d[COVARIATES].describe().loc[["mean", "std", "min", "max"]].round(3))
print("[Note] Log density equals log_store_count − log_area_size")
print(f"  Correlation check: "
      f"{np.corrcoef(np.log(d['store_density']), d['log_store_count']-d['log_area_size'])[0,1]:.6f}")

,log_store_count,log_foot_traffic,log_area_size,log_attracting_facilities
mean,2.584,13.657,11.634,3.187
std,0.849,1.016,0.842,0.995
min,1.609,2.197,7.525,0.000
max,6.522,15.969,14.717,6.389


[Note] Log density equals log_store_count − log_area_size
  Correlation check: 1.000000


### Covariate correlation

Check in advance whether multicollinearity in the propensity-score model reaches a problematic level.

In [9]:
display(d[COVARIATES + ["store_density", "food_service_share", "franchise_share"]].corr().round(3))

,log_store_count,log_foot_traffic,log_area_size,log_attracting_facilities,store_density,food_service_share,franchise_share
log_store_count,1.000,0.291,0.380,0.417,0.379,0.090,-0.131
log_foot_traffic,0.291,1.000,0.646,0.533,-0.273,-0.022,0.070
log_area_size,0.380,0.646,1.000,0.725,-0.461,-0.235,0.150
log_attracting_facilities,0.417,0.533,0.725,1.000,-0.245,-0.173,0.262
store_density,0.379,-0.273,-0.461,-0.245,1.000,0.278,-0.169
food_service_share,0.090,-0.022,-0.235,-0.173,0.278,1.000,-0.041
franchise_share,-0.131,0.070,0.150,0.262,-0.169,-0.041,1.000


## 3-7. Scoring frame for prediction ★

Rows for `SCORE_Q` (20261) have no next quarter, so no label can be created. They therefore drop out of the filter above — **but these rows are exactly the prediction targets.**

Only the label is missing; the features are intact, so we apply the **same filter and the same transformation** as the training panel and save them separately. If the transformation diverges here, the STEP 9 predictions will silently go wrong.

| Use | Feature quarter | Target quarter | Label |
|---|---|---|---|
| Training | `TRAIN_QUARTERS` (20231~20253) | 20232~20254 | Yes |
| Validation | `VALID_Q` (20254) | 20261 | Yes |
| Prediction | `SCORE_Q` (20261) | 20262 | No |

> ⚠️ **Putting `VALID_Q` (20254) into training would give a preview of the 20261 results.** `config.TRAIN_QUARTERS` already excludes it, so always use that constant in STEP 8.

In [10]:
score = df[(df["quarter_code"] == SCORE_Q)
           & (df["store_count"] >= MIN_STORES)
           & (df["foot_traffic_total"].notna())
           & (df["area_size"] > 0)].copy()
score = add_features(score).drop(columns=["y", "next_quarter_idx", "net_decline_now"])

print(f"Scoring frame (features {SCORE_Q} → target 20262)  {score.shape}")
print(f"  Confirm no label column: {'y' not in score.columns}")

# Check the columns match the training panel — if they diverge here, STEP 9 silently goes wrong
need = COVARIATES + ["store_density", "food_service_share", "franchise_share", "sales_observed",
                     "district", "area_type", "service_name"]
missing = [c for c in need if c not in score.columns]
print(f"  Missing required columns: {missing or 'none'}")
assert not missing
print(f"  Covariate missing: {dict(score[COVARIATES].isna().sum())}")

Scoring frame (features 20261 → target 20262)  (5428, 32)
  Confirm no label column: True
  Missing required columns: none
  Covariate missing: {'log_store_count': np.int64(0), 'log_foot_traffic': np.int64(0), 'log_area_size': np.int64(0), 'log_attracting_facilities': np.int64(0)}


### Time split summary

In [11]:
split = pd.DataFrame([
    {"purpose": "train", "feature_quarter": f"{TRAIN_QUARTERS[0]}~{TRAIN_QUARTERS[-1]}",
     "n_rows": int(d["quarter_code"].isin(TRAIN_QUARTERS).sum()),
     "positive_rate": d.loc[d["quarter_code"].isin(TRAIN_QUARTERS), "y"].mean()},
    {"purpose": "valid", "feature_quarter": str(VALID_Q),
     "n_rows": int((d["quarter_code"] == VALID_Q).sum()),
     "positive_rate": d.loc[d["quarter_code"] == VALID_Q, "y"].mean()},
    {"purpose": "pred", "feature_quarter": str(SCORE_Q), "n_rows": len(score), "positive_rate": np.nan},
])
display(split.round(3))
assert VALID_Q not in TRAIN_QUARTERS, "The validation quarter got mixed into training"
print("✅ The validation quarter is excluded from training")

,purpose,feature_quarter,n_rows,positive_rate
0,train,20231~20253,61996,0.254
1,valid,20254,5479,0.260
2,pred,20261,5428,NaN


✅ The validation quarter is excluded from training


## 3-8. Save

In [12]:
d = d.drop(columns=["next_quarter_idx"])
d.to_pickle(PROC / "03_panel.pkl")
mb = (PROC / "03_panel.pkl").stat().st_size / 1024**2
print(f"[Saved] 03_panel.pkl  {d.shape}  {mb:.1f} MB   ← training and validation")

score.to_pickle(PROC / "03_score.pkl")
mb2 = (PROC / "03_score.pkl").stat().st_size / 1024**2
print(f"[Saved] 03_score.pkl  {score.shape}  {mb2:.1f} MB   ← STEP 9 prediction target")
print(f"\nSample by commercial-area type")
display(d.groupby("area_type").agg(n_cells=("y", "size"), net_decline_rate=("y", "mean")).round(3))

[Saved] 03_panel.pkl  (67475, 34)  16.7 MB   ← training and validation


[Saved] 03_score.pkl  (5428, 32)  1.3 MB   ← STEP 9 prediction target

Sample by commercial-area type


,n_cells,net_decline_rate
area_type,,
neighborhood commercial area,34647,0.236
tourist special zone,696,0.378
developed commercial area,23011,0.292
traditional market,9121,0.224


---

## To do in STEP 4

- Skim the net decline rate by commercial-area type and by district (gu)
- Distribution of the new variables `franchise_share`, `food_service_share`, `store_density` and their relationship with y
- Tourist special zones have only 6 commercial areas, so no separate conclusion is drawn

STEP 8 (prediction) splits `03_panel.pkl` using `config.TRAIN_QUARTERS` / `VALID_Q`, and STEP 9 assigns probabilities to `03_score.pkl`.